# Week 04: Làm sạch dữ liệu và cleaning log

Mục tiêu hôm nay không phải là sửa dữ liệu cho đẹp. Mục tiêu là tạo **một cleaned dataset có thể tái lập**: giữ raw CSV, diagnose lỗi, chuẩn hóa nhãn, chuyển kiểu số, ghi cleaning log, export dữ liệu sạch và viết một note cho Methods/Data preparation.

## 1. Cài đặt ý tưởng

Raw data là dữ liệu gốc. Cleaned data là bảng dẫn xuất bằng code. Tuần này không sửa file raw bằng tay.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib

import pandas as pd

THIS_WEEK = "week-04-data-cleaning-log"
EXPECTED_SHA256 = "98c55be211cf92de8fae30343c7a1bb28d1eccf24bba5fcc6af44d777f7e0614"


def find_week_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "weeks" / THIS_WEEK,
        Path.cwd().parent,
        Path.cwd().parent / "weeks" / THIS_WEEK,
    ]
    for candidate in candidates:
        if candidate.name == THIS_WEEK and (candidate / "data/raw").exists():
            return candidate
        if (candidate / "data/raw/week04_messy_tcsol_scores.csv").exists():
            return candidate
    week_dir = Path.cwd() / "weeks" / THIS_WEEK
    week_dir.mkdir(parents=True, exist_ok=True)
    return week_dir


WEEK_DIR = find_week_dir()
DATA_PATH = WEEK_DIR / "data/raw/week04_messy_tcsol_scores.csv"
PROCESSED_DIR = WEEK_DIR / "data/processed"
TABLE_DIR = WEEK_DIR / "outputs/tables"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    source_url = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-04-data-cleaning-log/data/raw/week04_messy_tcsol_scores.csv"
    urlretrieve(source_url, DATA_PATH)

actual_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if actual_hash != EXPECTED_SHA256:
    raise ValueError("Downloaded CSV does not match the expected Week 04 teaching dataset.")

print("pandas version:", pd.__version__)
print("Week folder:", WEEK_DIR)
print("Data file:", DATA_PATH)


pandas version: 2.2.3
Week folder: /Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-04-data-cleaning-log
Data file: /Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-04-data-cleaning-log/data/raw/week04_messy_tcsol_scores.csv


## 2. Đọc raw CSV

Ta đọc dữ liệu với một danh sách `na_values` để pandas nhận ra các mã thiếu như `NA`, `missing`, `not recorded`, và ô trống.

In [2]:
missing_codes = ["NA", "missing", "not recorded", ""]
raw = pd.read_csv(DATA_PATH, na_values=missing_codes, keep_default_na=True)

print("Shape:", raw.shape)
print("Columns:", list(raw.columns))
print(raw.head(8).to_string(index=False))


Shape: (36, 8)
Columns: ['learner_id', 'class_group', 'activity_focus', 'pre_score', 'post_score', 'attendance_hours', 'completed', 'self_confidence']
learner_id class_group     activity_focus pre_score  post_score  attendance_hours completed  self_confidence
      S001           A      Measure Words        62        75.0               4.5       yes                4
      S002          a       measure words        58        70.0               4.0         Y                3
      S003     Class A      measure_words       NaN        72.0               4.5       yes                4
      S004           A      Measure words        60         NaN               4.0       yes                3
      S005           B result complements        55        68.0               3.5         y                2
      S006           b Result Complements        59        73.0               4.0       YES                3
      S007          B  result_complements        61        74.0               4.0     

## 3. Diagnose trước khi clean

Trước khi sửa gì, hãy xem missing values, kiểu dữ liệu, và nhãn đang lộn xộn thế nào.

In [3]:
print("Missing values by column:")
print(raw.isna().sum().to_string())

print("\nData types before cleaning:")
print(raw.dtypes.to_string())

print("\nRaw activity labels:")
print(raw["activity_focus"].value_counts(dropna=False).to_string())


Missing values by column:
learner_id          0
class_group         0
activity_focus      0
pre_score           4
post_score          3
attendance_hours    1
completed           1
self_confidence     0

Data types before cleaning:
learner_id           object
class_group          object
activity_focus       object
pre_score            object
post_score          float64
attendance_hours    float64
completed            object
self_confidence       int64

Raw activity labels:
activity_focus
Measure Words         4
Result Complements    3
Word Order            3
measure words         2
Vocabulary Review     2
vocabulary_review     2
Vocab Review          2
word-order            2
word_order            2
word order            2
result-complements    2
result_complements    2
result complements    2
measure_words         2
vocabulary review     1
Measure words         1
vocab_review          1
vocab review          1


## 4. Bắt đầu cleaning log

Cleaning log là bảng ghi quyết định: cột nào có vấn đề, mình quyết định gì, và vì sao.

In [4]:
cleaning_steps = []

def log_step(step, column, problem, decision, reason):
    cleaning_steps.append({
        "step": step,
        "column": column,
        "problem": problem,
        "decision": decision,
        "reason": reason,
    })

clean = raw.copy()
log_step(1, "all", "raw data may be overwritten", "created a cleaned copy", "preserve the original CSV")


## 5. Chuẩn hóa `class_group`

Các label như `a`, `Class A`, và `A` phải trở về một chuẩn chung để groupby không đếm thành nhiều nhóm giả.

In [5]:
clean["class_group_raw"] = clean["class_group"]
class_map = {
    "a": "A", "class a": "A",
    "b": "B", "class b": "B",
    "c": "C", "class c": "C",
    "d": "D", "class d": "D",
}
clean["class_group"] = (
    clean["class_group"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(class_map)
)
log_step(2, "class_group", "case, spaces, and Class A/Class B variants", "mapped to A-D", "standard labels make group summaries readable")

print(clean[["class_group_raw", "class_group"]].drop_duplicates().sort_values("class_group_raw").to_string(index=False))


class_group_raw class_group
              a           A
             a            A
              A           A
              B           B
             B            B
              C           C
        Class A           A
        Class B           B
        Class C           C
        Class D           D
              D           D
              b           B
              c           C
              d           D


## 6. Chuẩn hóa `activity_focus`

Đây là cột dùng cho bảng mô tả. Nếu không clean, `Measure Words`, `measure words`, và `measure_words` sẽ bị đếm riêng.

In [6]:
clean["activity_focus_raw"] = clean["activity_focus"]
activity_map = {
    "measure words": "measure_words",
    "result complements": "result_complements",
    "word order": "word_order",
    "vocab review": "vocabulary_review",
    "vocabulary review": "vocabulary_review",
}
activity_key = (
    clean["activity_focus"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
)
clean["activity_focus"] = activity_key.map(activity_map)
log_step(3, "activity_focus", "spelling, case, hyphen, and underscore variants", "mapped to four standard labels", "paper tables need stable group labels")

unmapped = clean[clean["activity_focus"].isna()]["activity_focus_raw"].dropna().unique()
print("Unmapped activity labels:", list(unmapped))
print(clean[["activity_focus_raw", "activity_focus"]].drop_duplicates().sort_values("activity_focus_raw").to_string(index=False))


Unmapped activity labels: []
activity_focus_raw     activity_focus
     Measure Words      measure_words
     Measure words      measure_words
Result Complements result_complements
      Vocab Review  vocabulary_review
 Vocabulary Review  vocabulary_review
        Word Order         word_order
     measure words      measure_words
     measure_words      measure_words
result complements result_complements
result-complements result_complements
result_complements result_complements
      vocab review  vocabulary_review
      vocab_review  vocabulary_review
 vocabulary review  vocabulary_review
 vocabulary_review  vocabulary_review
        word order         word_order
        word-order         word_order
        word_order         word_order


## 7. Chuẩn hóa `completed`

`yes`, `Y`, `YES` nên về `yes`; `n` nên về `no`; ô trống vẫn là missing.

In [7]:
clean["completed_raw"] = clean["completed"]
completed_map = {"yes": "yes", "y": "yes", "no": "no", "n": "no"}
clean["completed"] = (
    clean["completed"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(completed_map)
)
log_step(4, "completed", "yes/no variants and blank status", "mapped to yes/no; blanks remain missing", "the inclusion rule must be explicit")

print(clean[["completed_raw", "completed"]].drop_duplicates().to_string(index=False))


completed_raw completed
          yes       yes
            Y       yes
            y       yes
          YES       yes
           no        no
            n        no
          NaN       NaN


## 8. Chuyển cột số

Không thay missing bằng 0. `0` là một điểm thật; missing là chưa biết. Dùng `errors="coerce"` để giá trị không hợp lệ thành missing.

In [8]:
numeric_columns = ["pre_score", "post_score", "attendance_hours", "self_confidence"]
for column in numeric_columns:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")
log_step(5, ", ".join(numeric_columns), "numbers stored as text or invalid strings", "converted with pd.to_numeric(errors='coerce')", "invalid measurements should be missing, not guessed")

print(clean[numeric_columns].isna().sum().to_string())
print("\nData types after conversion:")
print(clean[numeric_columns].dtypes.to_string())


pre_score           5
post_score          3
attendance_hours    1
self_confidence     0

Data types after conversion:
pre_score           float64
post_score          float64
attendance_hours    float64
self_confidence       int64


## 9. Tạo `gain_score` và bảng usable rows

`gain_score` chỉ có nghĩa khi cả pre và post đều có số. Ta tạo flag để biết dòng nào dùng được cho summary pre/post.

In [9]:
clean = clean.assign(
    gain_score=clean["post_score"] - clean["pre_score"],
    usable_pre_post=lambda df: df["pre_score"].notna() & df["post_score"].notna() & (df["completed"] == "yes")
)
log_step(6, "gain_score", "derived measure needs complete pre/post scores", "created only where pre and post are numeric", "do not calculate gains from unknown scores")

print(clean["usable_pre_post"].value_counts().to_string())
print(clean[["learner_id", "activity_focus", "pre_score", "post_score", "gain_score", "usable_pre_post"]].head(10).to_string(index=False))


usable_pre_post
True     25
False    11
learner_id     activity_focus  pre_score  post_score  gain_score  usable_pre_post
      S001      measure_words       62.0        75.0        13.0             True
      S002      measure_words       58.0        70.0        12.0             True
      S003      measure_words        NaN        72.0         NaN            False
      S004      measure_words       60.0         NaN         NaN            False
      S005 result_complements       55.0        68.0        13.0             True
      S006 result_complements       59.0        73.0        14.0             True
      S007 result_complements       61.0        74.0        13.0             True
      S008 result_complements        NaN        71.0         NaN            False
      S009         word_order       64.0        73.0         9.0             True
      S010         word_order       63.0        72.0         9.0             True


## 10. Before/after summary

Bảng này cho thấy cleaning đã giảm số nhãn rời rạc và xác định số dòng usable cho phân tích.

In [10]:
raw_pre_numeric = pd.to_numeric(raw["pre_score"], errors="coerce")
raw_post_numeric = pd.to_numeric(raw["post_score"], errors="coerce")
raw_literal_completed = raw["completed"].astype("string").str.strip().str.lower() == "yes"
raw_usable_pre_post = raw_literal_completed & raw_pre_numeric.notna() & raw_post_numeric.notna()

cleaning_summary = pd.DataFrame([
    {
        "check": "activity_focus labels",
        "before": raw["activity_focus"].nunique(dropna=True),
        "after": clean["activity_focus"].nunique(dropna=True),
        "note": "standardized activity labels"
    },
    {
        "check": "class_group labels",
        "before": raw["class_group"].astype("string").str.strip().nunique(dropna=True),
        "after": clean["class_group"].nunique(dropna=True),
        "note": "standardized class sections"
    },
    {
        "check": "completed labels",
        "before": raw["completed"].astype("string").str.strip().str.lower().nunique(dropna=True),
        "after": clean["completed"].nunique(dropna=True),
        "note": "mapped yes/no variants"
    },
    {
        "check": "usable pre/post rows",
        "before": int(raw_usable_pre_post.sum()),
        "after": int(clean["usable_pre_post"].sum()),
        "note": "normalizing completed labels recovers usable rows"
    }
])
print(cleaning_summary.to_string(index=False))


                check  before  after                                              note
activity_focus labels      18      4                      standardized activity labels
   class_group labels      12      4                       standardized class sections
     completed labels       4      2                            mapped yes/no variants
 usable pre/post rows      19     25 normalizing completed labels recovers usable rows


## 11. Export cleaned artifacts

Xuất ba artifact: cleaned CSV, cleaning log, và before/after table. Đây là phần nộp bài chính của Week 04.

In [11]:
cleaning_log = pd.DataFrame(cleaning_steps)

cleaned_path = PROCESSED_DIR / "week04_cleaned_tcsol_scores.csv"
log_path = TABLE_DIR / "week04_cleaning_log.csv"
summary_path = TABLE_DIR / "week04_cleaning_summary.csv"

clean.to_csv(cleaned_path, index=False)
cleaning_log.to_csv(log_path, index=False)
cleaning_summary.to_csv(summary_path, index=False)


def display_path(path):
    try:
        return path.relative_to(Path.cwd())
    except ValueError:
        return path

print("Saved cleaned data to:", display_path(cleaned_path))
print("Saved cleaning log to:", display_path(log_path))
print("Saved cleaning summary to:", display_path(summary_path))


Saved cleaned data to: weeks/week-04-data-cleaning-log/data/processed/week04_cleaned_tcsol_scores.csv
Saved cleaning log to: weeks/week-04-data-cleaning-log/outputs/tables/week04_cleaning_log.csv
Saved cleaning summary to: weeks/week-04-data-cleaning-log/outputs/tables/week04_cleaning_summary.csv


## 12. Cleaning decision note cho Methods

Viết note ngắn, rõ quyết định và limitation. Không nói dữ liệu đã “hoàn hảo”.

In [12]:
usable_n = int(clean["usable_pre_post"].sum())
cleaning_note = (
    "Before analysis, the raw CSV was preserved and a cleaned copy was created in pandas. "
    "Class-group, activity-focus, and completion labels were standardized with predefined mappings. "
    "Score and attendance columns were converted to numeric values with pd.to_numeric(errors='coerce'), "
    "so invalid entries were treated as missing rather than replaced with zero. "
    f"After cleaning, {usable_n} records had completed status and numeric pre/post scores for a descriptive summary. "
    "The main limitation is that cleaning improves consistency but does not recover unknown scores or make the activity groups causal evidence."
)

print(cleaning_note)


Before analysis, the raw CSV was preserved and a cleaned copy was created in pandas. Class-group, activity-focus, and completion labels were standardized with predefined mappings. Score and attendance columns were converted to numeric values with pd.to_numeric(errors='coerce'), so invalid entries were treated as missing rather than replaced with zero. After cleaning, 25 records had completed status and numeric pre/post scores for a descriptive summary. The main limitation is that cleaning improves consistency but does not recover unknown scores or make the activity groups causal evidence.


## 13. Bài tập nhỏ trong notebook

Đổi mapping cho một label mới, ví dụ thêm `"measure word": "measure_words"`, rồi chạy lại từ section 6. Sau đó ghi: số unmapped labels có đổi không?